1. Build SUMO POIs from MATSim facilities used by the filtered population

In [1]:
import gzip
import xml.etree.ElementTree as ET
from pathlib import Path

from pyproj import CRS, Transformer

# ============================================================
# Build SUMO POIs from MATSim facilities used by the filtered population
# ============================================================

# ------------------------------
# CONFIG
# ------------------------------
EQASIM_DIR = Path("../Eqasim_output")  # Folder containing Eqasim/MATSim outputs

FAC_PATH = EQASIM_DIR / "17facilities.xml.gz"
POP_PATH = EQASIM_DIR / "17population_filtered.xml.gz"  # Filtered population XML (gz)

OUT_POI = Path("./activities_poi_lonlat.add.xml")

# MATSim coordinates CRS (France): Lambert-93 (EPSG:2154) -> WGS84 (EPSG:4326)
CRS_MATSIM = CRS.from_epsg(2154)
CRS_WGS84 = CRS.from_epsg(4326)
transformer = Transformer.from_crs(CRS_MATSIM, CRS_WGS84, always_xy=True)

if not FAC_PATH.exists():
    raise FileNotFoundError(f"Facilities file not found: {FAC_PATH.resolve()}")

if not POP_PATH.exists():
    raise FileNotFoundError(f"Filtered population file not found: {POP_PATH.resolve()}")

# ------------------------------
# 1) Read facility IDs used by the (filtered) population
# ------------------------------
def read_used_facility_ids(pop_gz_path: Path) -> set[str]:
    used: set[str] = set()

    with gzip.open(pop_gz_path, "rb") as f:
        # Streaming parse to keep memory usage low
        context = ET.iterparse(f, events=("start", "end"))
        _, root = next(context)

        for event, elem in context:
            if event == "start" and elem.tag == "activity":
                # Depending on MATSim export, the attribute can be 'facility' or 'facilityId'
                fid = elem.attrib.get("facility") or elem.attrib.get("facilityId")
                if fid:
                    used.add(fid)

            # Free memory progressively
            if event == "end":
                elem.clear()

        root.clear()

    return used


used_facilities = read_used_facility_ids(POP_PATH)
print(f"[INFO] Used facilities found in population: {len(used_facilities)}")

# ------------------------------
# 2) Parse facilities and write SUMO POIs (lon/lat) on the fly
# ------------------------------
poi_count = 0

with open(OUT_POI, "w", encoding="utf-8") as out:
    out.write('<?xml version="1.0" encoding="UTF-8"?>\n')
    out.write("<additional>\n")

    with gzip.open(FAC_PATH, "rb") as f:
        context = ET.iterparse(f, events=("start", "end"))
        _, root = next(context)

        current_id = None
        current_x = None
        current_y = None
        current_types: list[str] = []

        for event, elem in context:
            if event == "start" and elem.tag == "facility":
                fid = elem.attrib.get("id")
                if fid and fid in used_facilities:
                    current_id = fid
                    current_x = float(elem.attrib["x"])
                    current_y = float(elem.attrib["y"])
                    current_types = []
                else:
                    current_id = None  # skip this facility

            elif event == "start" and elem.tag == "activity" and current_id:
                t = elem.attrib.get("type")
                if t:
                    current_types.append(t)

            elif event == "end" and elem.tag == "facility":
                if current_id is not None:
                    lon, lat = transformer.transform(current_x, current_y)
                    poi_type = current_types[0] if current_types else "activity"
                    out.write(
                        f'  <poi id="{current_id}" lon="{lon:.6f}" lat="{lat:.6f}" type="{poi_type}"/>\n'
                    )
                    poi_count += 1

                # Reset + free memory
                current_id = None
                current_x = None
                current_y = None
                current_types = []
                elem.clear()

            elif event == "end":
                elem.clear()

        root.clear()

    out.write("</additional>\n")

print(f"[OK] POIs written: {poi_count}")
print(f"[OK] Output file: {OUT_POI}")

[INFO] Used facilities found in population: 16951
[OK] POIs written: 16951
[OK] Output file: activities_poi_lonlat.add.xml


2. Map POIs to SUMO network (multimode facilities mapping)

In [2]:
import os
import sys
import csv
from pathlib import Path
import xml.etree.ElementTree as ET

# ============================================================
# Map POIs to SUMO network (multimode facilities mapping)
# ============================================================

# ------------------------------
# CONFIG
# ------------------------------
POI_XML = Path("./activities_poi_lonlat.add.xml")  # SUMO POIs (lon/lat or x/y)
OUT_DIR = Path("./")
NET_DIR = Path("../1-network")
NET_FILE = "cda_la_rochelle.net.xml"

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "facilities2sumo_multimode.csv"

MODES = ["pedestrian", "passenger"]  # SUMO vClass
SEARCH_DEPTH = 2                     # incoming/outgoing connectivity checks depth
MIN_EDGE_LEN = 5.0                   # avoid micro-edges
INITIAL_RADIUS = 20                  # meters
MAX_RADIUS = 4000                    # meters
RADIUS_MULT = 2

# ------------------------------
# SUMO / sumolib
# ------------------------------
def require_sumolib():
    sumo_home = os.environ.get("SUMO_HOME")
    if not sumo_home:
        raise RuntimeError(
            'SUMO_HOME is not set. Example (Windows): setx SUMO_HOME "C:\\Program Files\\Eclipse\\Sumo"\n'
            'Example (Linux): export SUMO_HOME=/usr/share/sumo'
        )
    tools = os.path.join(sumo_home, "tools")
    if tools not in sys.path:
        sys.path.append(tools)
    import sumolib  # type: ignore
    return sumolib

sumolib = require_sumolib()

net_path = NET_DIR / NET_FILE
if not net_path.exists():
    raise FileNotFoundError(f"SUMO network file not found: {net_path.resolve()}")

if not POI_XML.exists():
    raise FileNotFoundError(f"POI file not found: {POI_XML.resolve()}")

print(f"[INFO] Reading network: {net_path}")
net = sumolib.net.readNet(str(net_path), withInternal=False)

# ------------------------------
# POI iterator (supports lon/lat OR x/y)
# ------------------------------
def iter_pois(poi_xml_path: Path):
    tree = ET.parse(poi_xml_path)
    root = tree.getroot()
    for poi in root.iter("poi"):
        pid = poi.attrib.get("id")
        ptype = poi.attrib.get("type", "")

        if "lon" in poi.attrib and "lat" in poi.attrib:
            lon = float(poi.attrib["lon"])
            lat = float(poi.attrib["lat"])
            yield pid, ptype, lat, lon, None, None
        elif "x" in poi.attrib and "y" in poi.attrib:
            x = float(poi.attrib["x"])
            y = float(poi.attrib["y"])
            yield pid, ptype, None, None, x, y

# ------------------------------
# Robust mapping helpers
# ------------------------------
def lane_pos_on_lane(lane_obj, x, y):
    lanePos, _ = sumolib.geomhelper.polygonOffsetAndDistanceToPoint((x, y), lane_obj.getShape())
    L = lane_obj.getLength()
    lanePos = max(0.1, min(lanePos, L - 0.1))
    return lanePos

def find_valid_incoming(lane, vclass, depth):
    current_lanes = [lane]
    for _ in range(depth):
        next_lanes = []
        for ln in current_lanes:
            for inc in ln.getIncoming():
                if inc.allows(vclass):
                    next_lanes.append(inc)
        if not next_lanes:
            return False
        current_lanes = next_lanes
    return True

def find_valid_outgoing(lane, vclass, depth):
    current_lanes = [lane]
    for _ in range(depth):
        next_lanes = []
        for ln in current_lanes:
            for out in ln.getOutgoingLanes():
                if out.allows(vclass):
                    next_lanes.append(out)
        if not next_lanes:
            return False
        current_lanes = next_lanes
    return True

def check_edge(edge, vclass, depth):
    for lane in edge.getLanes():
        if lane.allows(vclass):
            if find_valid_incoming(lane, vclass, depth) and find_valid_outgoing(lane, vclass, depth):
                return True
    return False

def map_point_to_edge_lane_pos(net, x, y, vclass, depth=2):
    radius = INITIAL_RADIUS
    while radius <= MAX_RADIUS:
        neigh = net.getNeighboringEdges(x, y, radius)
        if not neigh:
            radius *= RADIUS_MULT
            continue

        candidates = sorted([(dist, edge) for edge, dist in neigh], key=lambda t: t[0])

        for dist, edge in candidates:
            if edge.getFunction() == "internal":
                continue
            if edge.getLength() < MIN_EDGE_LEN:
                continue
            if not edge.allows(vclass):
                continue
            if not check_edge(edge, vclass, depth):
                continue

            lane_obj = None
            for ln in edge.getLanes():
                if ln.allows(vclass):
                    lane_obj = ln
                    break
            if lane_obj is None:
                continue

            pos = lane_pos_on_lane(lane_obj, x, y)
            return edge.getID(), lane_obj.getID(), pos, dist

        radius *= RADIUS_MULT

    raise RuntimeError(f"No valid edge found for vclass={vclass} around (x={x}, y={y})")

def clean_error_message(e: Exception) -> str:
    """Make exception message CSV-friendly and compact."""
    msg = str(e).replace("\r", " ").replace("\n", " | ").strip()
    return msg[:500]  # keep it short-ish

# ------------------------------
# RUN
# ------------------------------
n = 0
fail = {m: 0 for m in MODES}

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)

    # Added: status + error_reason
    w.writerow([
        "poi_id", "poi_type", "lat", "lon", "x", "y", "mode",
        "edge_id", "lane_id", "pos", "dist_to_edge",
        "status", "error_reason"
    ])

    for pid, ptype, lat, lon, x_in, y_in in iter_pois(POI_XML):
        n += 1

        # Convert to SUMO network XY if the POI is in lon/lat
        if lat is not None and lon is not None:
            x, y = net.convertLonLat2XY(lon, lat)
        else:
            x, y = x_in, y_in

        for mode in MODES:
            try:
                edge_id, lane_id, pos, dist = map_point_to_edge_lane_pos(net, x, y, mode, depth=SEARCH_DEPTH)
                w.writerow([
                    pid, ptype, lat, lon, f"{x:.2f}", f"{y:.2f}", mode,
                    edge_id, lane_id, f"{pos:.2f}", f"{dist:.2f}",
                    "mapped", ""
                ])
            except Exception as e:
                fail[mode] += 1
                reason = clean_error_message(e)
                w.writerow([
                    pid, ptype, lat, lon, f"{x:.2f}", f"{y:.2f}", mode,
                    "", "", "", "",
                    "failed", reason
                ])
                if fail[mode] <= 10:
                    print(f"[WARN] POI not mapped (id={pid}, mode={mode}): {reason}")

print("\n[DONE]")
print(f"[INFO] POIs read: {n}")
for m in MODES:
    print(f"[INFO] Failures ({m}): {fail[m]}")
print(f"[INFO] Output CSV: {OUT_CSV}")

[INFO] Reading network: ..\1-network\cda_la_rochelle.net.xml
[WARN] POI not mapped (id=edu_502, mode=pedestrian): No valid edge found for vclass=pedestrian around (x=44838.224948720075, y=-14873.579033639282)
[WARN] POI not mapped (id=edu_502, mode=passenger): No valid edge found for vclass=passenger around (x=44838.224948720075, y=-14873.579033639282)
[WARN] POI not mapped (id=sec_10221, mode=pedestrian): No valid edge found for vclass=pedestrian around (x=98490.74474445404, y=13645.747533343732)
[WARN] POI not mapped (id=sec_10221, mode=passenger): No valid edge found for vclass=passenger around (x=98490.74474445404, y=13645.747533343732)
[WARN] POI not mapped (id=sec_21248, mode=pedestrian): No valid edge found for vclass=pedestrian around (x=43093.83750561974, y=-14476.920206519775)
[WARN] POI not mapped (id=sec_21248, mode=passenger): No valid edge found for vclass=passenger around (x=43093.83750561974, y=-14476.920206519775)
[WARN] POI not mapped (id=sec_25210, mode=pedestrian): 

In [5]:
import pandas as pd

# ============================================================
# Load facilities-to-SUMO mapping (poi_id + mode)
# ============================================================

MAP_CSV = "./facilities2sumo_multimode.csv"

# NOTE: your pipeline writes ';' separated CSV files
df = pd.read_csv(MAP_CSV, sep=",")

# If the mapping script wrote status/error_reason columns, keep only successful rows
if "status" in df.columns:
    df = df[df["status"] == "mapped"].copy()

# Build lookup: (poi_id, mode) -> (edge_id, pos)
map_access = {}

for _, r in df.iterrows():
    poi_id = str(r["poi_id"])
    mode = str(r["mode"])  # pedestrian / passenger
    edge_id = r["edge_id"]
    pos = r["pos"]

    if isinstance(edge_id, str) and edge_id.strip() != "" and pd.notna(pos):
        map_access[(poi_id, mode)] = (edge_id, float(pos))

print(f"[INFO] Available mappings: {len(map_access)}")


[INFO] Available mappings: 33888
